# Train YOLO11n — Broiler Pathology Detector (end-to-end, Google Colab)

**Project:** A Smart Poultry Health Monitoring and Management System for Smallholder Farmers.  
**Methodology:** Section 3.9. **Dataset:** Elmessery et al. (2023), *Agriculture* 13(8):1527, RGB subset only.

This notebook runs the whole ML pipeline in Colab: environment setup -> GPU check -> dataset
acquisition -> YOLO-format prep -> EDA -> leakage-safe split -> transfer-learning training ->
iterative diagnosis -> evaluation -> export. It calls the audited `ml/src` modules so behaviour
matches the command-line tools exactly.

> **Honesty contract.** Nothing in this notebook is pre-filled with results. Every number, plot
> and metric is produced by executing a cell against the real dataset. If a cell has not been run,
> the corresponding dissertation figure stays a pending placeholder.

**Before you start (one-time, manual — Claude cannot click these for you):**
1. Runtime -> Change runtime type -> **GPU** (T4 is enough).
2. You will be asked to authorise Google Drive access when the mount cell runs.

## 1. Environment setup & GPU check

In [ ]:
# Install pinned deps. Ultralytics pulls a compatible torch on Colab.
%pip -q install ultralytics==8.3.0 gdown==5.2.0 pyyaml==6.0.2 matplotlib==3.9.2

In [ ]:
import torch, platform, subprocess
print('Python :', platform.python_version())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print(subprocess.getoutput('nvidia-smi | head -n 15'))
else:
    print('WARNING: no GPU runtime. Training will be very slow.')
    print('Runtime -> Change runtime type -> Hardware accelerator -> GPU, then re-run.')

## 2. Get the project code into Colab

Two options. **A** — clone from your Git remote (recommended, keeps `src/` in sync).
**B** — mount Drive and point `ML` at a copy you uploaded. Edit `ML` to your actual path.

In [ ]:
import os, sys, pathlib

# --- Option A: clone (uncomment and set your repo URL) ---
# !git clone https://github.com/<you>/poultry-monitoring-system.git /content/pms
# ML = '/content/pms/ml'

# --- Option B: mount Drive and use an uploaded copy ---
from google.colab import drive
drive.mount('/content/drive')
ML = '/content/drive/MyDrive/poultry-monitoring-system/ml'   # <-- EDIT to your path

assert os.path.isdir(ML), f'ML path not found: {ML}. Edit the ML variable above.'
os.chdir(ML)
sys.path.insert(0, ML)
print('Working dir:', os.getcwd())
print('src present:', os.path.isdir('src'))

## 3. Acquire the Elmessery RGB dataset

The dataset lives in a **public Google Drive folder** with ~10,000 JPGs. Reliable acquisition:

- **Preferred (large folders):** open the [shared folder](https://drive.google.com/drive/folders/1jj9LKL0d1YDyDez8xrmKWRWd3psFoeZ2),
  click **Add shortcut to Drive**, then copy it from your mounted Drive below. `gdown --folder`
  is capped at **50 files** and will silently truncate a 10k-image folder, so do not rely on it
  for the full set.
- **If the authors publish a single ZIP mirror**, `gdown` that file id instead (no 50-file cap on single files).

Set `RAW` to where the **RGB images** should land. We import only the visual subset (thermal excluded).

In [ ]:
DRIVE_FOLDER_ID = '1jj9LKL0d1YDyDez8xrmKWRWd3psFoeZ2'
RAW = os.path.join(ML, 'data', 'raw', 'elmessery_rgb')
os.makedirs(RAW, exist_ok=True)

# --- Path 1: copy from a shortcut you added to your Drive (recommended) ---
# SRC = '/content/drive/MyDrive/<shortcut-folder-name>'
# !cp -rn "$SRC"/. "$RAW"/

# --- Path 2: single-ZIP mirror (if available) ---
# import gdown; gdown.download(id='<ZIP_FILE_ID>', output='/content/elmessery.zip', quiet=False)
# !unzip -q -o /content/elmessery.zip -d "$RAW"

# --- Path 3: gdown folder (ONLY a <=50-file sample; NOT the full dataset) ---
# import gdown; gdown.download_folder(id=DRIVE_FOLDER_ID, output=RAW, quiet=False, use_cookies=False)

import glob
imgs = glob.glob(os.path.join(RAW, '**', '*.jpg'), recursive=True)
imgs += glob.glob(os.path.join(RAW, '**', '*.jpeg'), recursive=True)
imgs += glob.glob(os.path.join(RAW, '**', '*.png'), recursive=True)
print(f'RGB images found under RAW: {len(imgs)}')
assert imgs, 'No images imported yet. Use one of the acquisition paths above before continuing.'

### 3b. Record provenance (measured counts, licence to confirm)
Writes `data/metadata/elmessery_rgb_provenance.json` with a real file count + checksums.

In [ ]:
!python -m src.data.import_dataset --source "$RAW" --name elmessery_rgb \
    --licence 'CC-BY-4.0-article; dataset-licence-TO-CONFIRM' --no-copy

## 4. Inspect structure & convert annotations to YOLO format (if needed)

We do not assume the label format. This cell reports what is actually on disk so you can pick the
right conversion. YOLO expects, per image `x.jpg`, a `x.txt` with `class cx cy w h` (normalised).
If the download ships VOC XML / COCO JSON / classification folders, convert here before splitting.

In [ ]:
import collections, os
exts = collections.Counter()
for root, _, files in os.walk(RAW):
    for f in files:
        exts[os.path.splitext(f)[1].lower()] += 1
print('File types under RAW:')
for e, n in exts.most_common():
    print(f'  {e or "(none)"}: {n}')
print()
txts = [f for r,_,fs in os.walk(RAW) for f in fs if f.endswith('.txt')]
print('YOLO-style .txt labels present:', len(txts))
print('If 0 and labels are XML/JSON, add a conversion step here before Section 5.')
print('Ultralytics ships converters, e.g.:  from ultralytics.data.converter import convert_coco')

## 5. Deduplicate near-identical frames
Frames pulled from the same short capture can be near-duplicates; dropping them reduces leakage
and over-counting. Uses the audited perceptual-hash tool.

In [ ]:
os.makedirs('data/interim', exist_ok=True)
# Both tools take --dir and print a JSON report to stdout; tee to a file for the record.
!python -m src.data.verify_images --dir "$RAW" | tee data/interim/verify_images.json
!python -m src.data.find_duplicates --dir "$RAW" | tee data/interim/duplicates.json
print('Review duplicates.json and remove near-identical frames before splitting.')

## 6. Exploratory Data Analysis (produces the REAL Fig 3.12 / 3.13 assets)

Runs the same measurement code as `notebooks/01_dataset_eda.ipynb`. Saves plots into
`outputs/eda/`. **These PNGs are what replace the Fig 3.12 (class distribution) and Fig 3.13
(EDA) placeholders in the dissertation** — only after this cell has run on the real data.

In [ ]:
import os, json
from src.data.generate_statistics import compute
from pathlib import Path

IMAGES = Path('data/interim/images') if Path('data/interim/images').exists() else Path(RAW)
LABELS = Path('data/interim/labels') if Path('data/interim/labels').exists() else IMAGES
os.makedirs('outputs/eda', exist_ok=True)
stats = compute(IMAGES, LABELS, num_classes=6)
Path('outputs/eda/stats_raw.json').write_text(json.dumps(stats, indent=2))
print(json.dumps(stats, indent=2))

In [ ]:
# Class-distribution + bbox/size plots from measured stats. Class names from the shared mapping.
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.common.class_mapping import trained_classes
names = trained_classes()
dist = stats.get('class_distribution', {})
counts = [dist.get(str(i), 0) for i in range(len(names))]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(names, counts, color='#2e7d32')
ax.set_ylabel('Annotated instances'); ax.set_title('Class distribution (measured)')
ax.set_xticklabels(names, rotation=25, ha='right')
for i, c in enumerate(counts):
    ax.text(i, c, str(c), ha='center', va='bottom', fontsize=9)
fig.tight_layout(); fig.savefig('outputs/eda/fig_3_12_class_distribution.png', dpi=200)
print('Saved outputs/eda/fig_3_12_class_distribution.png')
print('Total instances:', sum(counts), '| classes with zero data:', [names[i] for i,c in enumerate(counts) if c==0])

## 7. Leakage-safe train/val/test split
Frames from the same capture group stay together (no leakage). Grouping by parent dir by default;
switch to `filename_prefix` if the images are flat with encoded ids.

In [ ]:
!python -m src.data.split_dataset \
    --images data/interim/images --labels data/interim/labels \
    --group-by parent_dir --train 0.7 --val 0.15 --test 0.15 --seed 42
!python -m src.data.create_dataset_yaml --out data/processed/dataset.yaml
!cat data/processed/dataset.yaml

## 8. Baseline training (transfer learning from yolo11n.pt)
Config comes from `configs/train.yaml` (conservative, poultry-realistic augmentation). Checkpoints
and curves are written by Ultralytics into `outputs/runs/broiler_yolo11n/`.

In [ ]:
os.environ['DATASET_YAML'] = os.path.join(ML, 'data/processed/dataset.yaml')
os.environ['PROJECT_DIR']  = os.path.join(ML, 'outputs/runs')
!python -m src.training.train_yolo --config configs/train.yaml

### 8b. Where the real training curves come from
Ultralytics writes `results.png` (loss + metric curves), `PR_curve.png`, `confusion_matrix.png`
into the run folder. Those are the **Fig 3.15 / 3.16 / 3.17** assets — copy them to `results/`
after this run finishes. Do **not** fill those figures until this cell has actually completed.

In [ ]:
import glob, shutil, os
runs = sorted(glob.glob('outputs/runs/broiler_yolo11n*'), key=os.path.getmtime)
assert runs, 'No run folder yet — training has not produced outputs.'
run = runs[-1]; print('Latest run:', run)
os.makedirs('results', exist_ok=True)
wanted = {'results.png':'fig_3_15_training_curves.png',
          'PR_curve.png':'fig_3_16_pr_curve.png',
          'confusion_matrix.png':'fig_3_17_confusion_matrix.png'}
for src_name, dst in wanted.items():
    p = os.path.join(run, src_name)
    if os.path.exists(p):
        shutil.copy(p, os.path.join('results', dst)); print('copied', dst)
    else:
        print('NOT YET PRESENT:', src_name, '(figure stays pending)')

## 9. Iterative diagnosis & hyperparameter tuning
Baseline -> read val metrics -> adjust -> retrain. Change one lever at a time and log why.
Levers: `imgsz` (640/800), `batch` (8/16/32), `lr0`, `epochs`, `patience`, augmentation strengths,
and later the inference `conf` threshold. Keep a short note per experiment.

In [ ]:
from ultralytics import YOLO
# Example second iteration — larger image size + gentler augmentation. Edit deliberately.
exp = YOLO('yolo11n.pt').train(
    data='data/processed/dataset.yaml',
    epochs=120, imgsz=800, batch=16, lr0=0.008, patience=25, seed=42,
    project='outputs/runs', name='broiler_yolo11n_imgsz800',
    fliplr=0.5, flipud=0.0, hsv_v=0.15, hsv_s=0.15, degrees=5.0, scale=0.3, mosaic=0.4, mixup=0.0,
)
# Compare val mAP50 / mAP50-95 across runs before choosing best.pt.

## 10. Evaluate on the held-out TEST split
Test-set metrics only — never tune on test. Writes `outputs/evaluation/*.json`.

In [ ]:
BEST = f'{run}/weights/best.pt'   # or the better iteration's best.pt
!python -m src.evaluation.evaluate_detection --weights "$BEST" \
    --data data/processed/dataset.yaml --split test

## 11. Export & quantize for Raspberry Pi (Fig-independent artefacts)
TFLite / ONNX / NCNN with quantisation. Record resulting size; benchmark latency on the **Pi**
(or CPU here, clearly labelled) with `src.deployment.benchmark`.

In [ ]:
for fmt in ['onnx', 'tflite', 'ncnn']:
    print('=== exporting', fmt, '===')
    !python -m src.deployment.export_model --weights "$BEST" --format {fmt} --imgsz 640 || true
print()
!ls -lh {run}/weights/ || true

In [ ]:
# CPU latency benchmark (label clearly as CPU, not Pi, if run in Colab).
!python -m src.deployment.benchmark --weights "$BEST" --runs 50 --imgsz 640 || true
print('NOTE: numbers above are Colab-CPU/GPU, not Raspberry Pi. Re-run on the Pi for edge figures.')

## 12. Record the model card
After you settle on `best.pt`, fill `ml/models/MODEL_CARD.md` with the run id, dataset version,
class list, measured test metrics, export sizes and benchmark host. That file — not this notebook —
is the citable record of the trained model.

In [ ]:
print('Checklist before writing MODEL_CARD.md:')
for step in ['Dataset provenance recorded (data/metadata/*.json)',
             'EDA plots in outputs/eda/',
             'Chosen run + best.pt identified',
             'Test-split metrics in outputs/evaluation/',
             'Exports + sizes recorded',
             'Benchmark host noted (Pi vs CPU)']:
    print('  [ ]', step)